# Phase 05G: Sequence-KD Seed-42 Ablation Gate

Notebook CPU này so sánh R0/R1/R2/R3 bằng Validation và áp dụng đúng gate đã
khóa ở Phase 05C. Không mở Test, không huấn luyện và không thay đổi phương pháp.

- primary: CIDEr-D;
- tie-break: BLEU-4, CE, epoch sớm;
- candidate phải hơn R1 ít nhất `+0.001 CIDEr-D`;
- BLEU-4 không được giảm quá `0.002`;
- nếu cả R2/R3 thất bại: giữ R1 và kết thúc Phase 5.

## Inputs

1. Phase 05C fallback entry.
2. Phase 04D R1 seed42 full run.
3. Phase 04F R1 three-seed aggregate.
4. Phase 05E R2 seed42 full run.
5. Phase 05F R3 seed42 full run.

CPU, Internet Off, không MS COCO/model/GPU.


## Cell 1 — Discover immutable inputs


In [1]:
import csv, hashlib, json, os, time
from datetime import datetime, timezone
from pathlib import Path
STARTED=time.perf_counter()
INPUT_ROOT=Path(os.environ.get("KLCN_INPUT_ROOT","/kaggle/input"))
NR=INPUT_ROOT/"notebooks" if (INPUT_ROOT/"notebooks").is_dir() else INPUT_ROOT
OUT=Path(os.environ.get("KLCN_OUTPUT_ROOT","/kaggle/working//phase5_sequence_kd_seed42_ablation_gate"))
AD=OUT/"audits"; RD=OUT/"results"; MD=OUT/"manifests"
for d in [OUT,AD,RD,MD]: d.mkdir(parents=True,exist_ok=True)
def now(): return datetime.now(timezone.utc).isoformat()
def read(p):
    with Path(p).open("r",encoding="utf-8") as h:return json.load(h)
def write(p,x):
    p=Path(p);t=p.with_suffix(p.suffix+".tmp");t.write_text(json.dumps(x,ensure_ascii=False,indent=2)+"\n",encoding="utf-8");os.replace(t,p)
def sha(p):
    d=hashlib.sha256()
    with Path(p).open("rb") as h:
        for b in iter(lambda:h.read(8*1024*1024),b""):d.update(b)
    return d.hexdigest()
def one(n):
    m=sorted(NR.rglob(n));assert len(m)==1,(n,m);return m[0].resolve()
def inside(root,rel):
    root=Path(root).resolve();p=(root/rel).resolve();assert p.is_relative_to(root);return p
C_PATH=one("PHASE5C_SEQUENCE_KD_FALLBACK_ENTRY_COMPLETE.json")
R1_PATH=one("PHASE4_TEACHER_A_S1_SEQUENCE_KD_SEED42_FULL_RUN_COMPLETE.json")
R1A_PATH=one("PHASE4_TEACHER_A_S1_SEQUENCE_KD_THREE_SEED_AGGREGATE_COMPLETE.json")
R2_PATH=one("PHASE5_R2_SEED42_FULL_RUN_COMPLETE.json")
R3_PATH=one("PHASE5_R3_SEED42_FULL_RUN_COMPLETE.json")
C,R1,R1A,R2,R3=map(read,[C_PATH,R1_PATH,R1A_PATH,R2_PATH,R3_PATH])
assert C["status"]=="complete" and C["feature_branch"]=="stopped_incompatible"
assert R1["status"]==R2["status"]==R3["status"]=="complete"
assert R1A["status"]=="complete" and R1A["reproduction_status"]=="passed"
for x in [C,R1,R1A,R2,R3]:
    assert x["test_set_status"]=="sealed" and x["test_set_used"] is False
write(AD/"phase5g_cell01_inputs.json",{"schema_version":"phase5g_cell01_v1","created_at_utc":now(),
 "status":"passed","inputs":"5 / 5","dataset_read":False,"training_started":False,
 "test_set_status":"sealed","test_set_used":False,"elapsed_seconds":time.perf_counter()-STARTED})
print("="*72);print("PHASE 05G CELL 1 STATUS");print("="*72)
print("Status                    : PASSED")
print("Inputs                    : 5 / 5 FOUND")
print("Training / Test           : NOT STARTED / SEALED")
print("Progress                  : 1 / 5 code cells")


PHASE 05G CELL 1 STATUS
Status                    : PASSED
Inputs                    : 5 / 5 FOUND
Training / Test           : NOT STARTED / SEALED
Progress                  : 1 / 5 code cells


## Cell 2 — Manifest and checkpoint integrity


In [2]:
STARTED=time.perf_counter()
def verify(root,marker):
    ref=marker["manifest"]["csv"];p=inside(root,ref["relative_path"]);assert sha(p)==ref["sha256"]
    with p.open("r",encoding="utf-8",newline="") as h:rows=list(csv.DictReader(h))
    assert len(rows)==int(marker["manifest_rows"])
    for r in rows:
        a=inside(root,r["relative_path"]);assert a.is_file()
        assert a.stat().st_size==int(r["size_bytes"]) and sha(a)==r["sha256"]
    return rows
r1rows=verify(R1_PATH.parent,R1);r2rows=verify(R2_PATH.parent,R2);r3rows=verify(R3_PATH.parent,R3)
for root,m in [(R1_PATH.parent,R1),(R2_PATH.parent,R2),(R3_PATH.parent,R3)]:
    b=inside(root,m["best_checkpoint"]["relative_path"]);assert sha(b)==m["best_checkpoint"]["sha256"]
assert R1["seed"]==R2["seed"]==R3["seed"]==42
write(AD/"phase5g_cell02_integrity.json",{"schema_version":"phase5g_cell02_v1","created_at_utc":now(),
 "status":"passed","manifest_rows_verified":len(r1rows)+len(r2rows)+len(r3rows),
 "best_checkpoints":"3 / 3 verified","paired_seed":42,"test_set_status":"sealed","test_set_used":False,
 "elapsed_seconds":time.perf_counter()-STARTED})
print("="*72);print("PHASE 05G CELL 2 STATUS");print("="*72)
print("Manifest rows verified    :",len(r1rows)+len(r2rows)+len(r3rows))
print("Best checkpoints          : 3 / 3 VERIFIED")
print("Paired seed               : 42")
print("Progress                  : 2 / 5 code cells")


PHASE 05G CELL 2 STATUS
Manifest rows verified    : 153
Best checkpoints          : 3 / 3 VERIFIED
Paired seed               : 42
Progress                  : 2 / 5 code cells


## Cell 3 — Build the locked R0/R1/R2/R3 comparison


In [3]:
STARTED=time.perf_counter()
contract=read(inside(C_PATH.parent,"configs/phase5c_sequence_kd_weight_ablation_contract.json"))
assert contract["status"]=="locked";gate=contract["gate_vs_r1"]
def row(arm,h,t,m):
    b=m["best_result"]
    return {"arm":arm,"human_weight":h,"teacher_weight":t,"CIDEr-D":float(b["metrics"]["CIDEr-D"]),
            "BLEU-4":float(b["metrics"]["BLEU-4"]),"METEOR":float(b["metrics"]["METEOR"]),
            "ROUGE-L":float(b["metrics"]["ROUGE-L"]),"CE":float(b["teacher_forced_ce"]),
            "best_epoch":int(b["epoch"])}
r0={"arm":"R0","human_weight":1.0,"teacher_weight":0.0,
    "CIDEr-D":float(R1["comparison"]["baseline_seed42"]["CIDEr-D"]),
    "BLEU-4":float(R1["comparison"]["baseline_seed42"]["BLEU-4"]),
    "METEOR":None,"ROUGE-L":None,"CE":float(R1["comparison"]["baseline_seed42"]["validation_ce"]),"best_epoch":None}
rows=[r0,row("R1",0.5,0.5,R1),row("R2",0.75,0.25,R2),row("R3",0.25,0.75,R3)]
r1=rows[1]
for r in rows[2:]:
    r["delta_cider_vs_r1"]=r["CIDEr-D"]-r1["CIDEr-D"]
    r["delta_bleu4_vs_r1"]=r["BLEU-4"]-r1["BLEU-4"]
    r["delta_ce_vs_r1"]=r["CE"]-r1["CE"]
    r["cider_gate"]=r["delta_cider_vs_r1"]>=float(gate["cider_delta_minimum"])
    r["bleu4_gate"]=r["delta_bleu4_vs_r1"]>=float(gate["bleu4_non_regression_tolerance"])
    r["gate_passed"]=r["cider_gate"] and r["bleu4_gate"]
csvp=RD/"phase5g_seed42_ablation.csv"
fields=list(rows[2].keys())
with csvp.open("w",encoding="utf-8",newline="") as h:
    w=csv.DictWriter(h,fieldnames=fields,extrasaction="ignore");w.writeheader();w.writerows(rows)
write(RD/"phase5g_seed42_ablation.json",{"schema_version":"phase5g_ablation_v1",
 "created_at_utc":now(),"rows":rows,"gate_vs_r1":gate,"selection_split":"validation",
 "test_set_status":"sealed","test_set_used":False})
write(AD/"phase5g_cell03_metrics.json",{"schema_version":"phase5g_cell03_v1","created_at_utc":now(),
 "status":"passed","rows":rows,"test_set_status":"sealed","test_set_used":False,
 "elapsed_seconds":time.perf_counter()-STARTED})
print("="*72);print("PHASE 05G CELL 3 — SEED42 METRICS");print("="*72)
for r in rows:print(r["arm"],"| CIDEr",round(r["CIDEr-D"],6),"| BLEU4",round(r["BLEU-4"],6),"| CE",round(r["CE"],6))
print("R2 gate                  :",rows[2]["gate_passed"])
print("R3 gate                  :",rows[3]["gate_passed"])
print("Progress                  : 3 / 5 code cells")


PHASE 05G CELL 3 — SEED42 METRICS
R0 | CIDEr 0.991791 | BLEU4 0.313856 | CE 2.214854
R1 | CIDEr 1.102155 | BLEU4 0.345204 | CE 2.355077
R2 | CIDEr 1.098302 | BLEU4 0.34617 | CE 2.262284
R3 | CIDEr 1.08615 | BLEU4 0.340002 | CE 2.570678
R2 gate                  : False
R3 gate                  : False
Progress                  : 3 / 5 code cells


## Cell 4 — Apply preregistered decision rule


In [4]:
STARTED=time.perf_counter()
candidates=[r for r in rows[2:] if r["gate_passed"]]
if candidates:
    candidates.sort(key=lambda r:(-r["CIDEr-D"],-r["BLEU-4"],r["CE"],r["best_epoch"]))
    winner=candidates[0];decision="candidate_passed";confirmation=True
else:
    winner=r1;decision="retain_R1";confirmation=False
assert decision=="retain_R1"
assert winner["arm"]=="R1"
decision_payload={"schema_version":"phase5g_seed42_gate_decision_v1","created_at_utc":now(),
 "status":"locked","decision":decision,"selected_arm":winner["arm"],
 "reason":"R2 and R3 failed the preregistered CIDEr-D improvement gate versus R1",
 "r2_delta_cider_vs_r1":rows[2]["delta_cider_vs_r1"],
 "r3_delta_cider_vs_r1":rows[3]["delta_cider_vs_r1"],
 "new_confirmation_runs_authorized":confirmation,
 "existing_r1_three_seed_aggregate_available":True,
 "phase5_conclusion":"No Sequence-KD weight improvement over alpha=0.5 at seed 42; retain reproduced R1.",
 "test_set_status":"sealed","test_set_used":False}
DECISION=RD/"phase5g_decision.json";write(DECISION,decision_payload)
write(AD/"phase5g_cell04_gate.json",{"schema_version":"phase5g_cell04_v1","created_at_utc":now(),
 "status":"passed","selected_arm":"R1","new_confirmation_runs_authorized":False,
 "post_hoc_override":False,"test_set_status":"sealed","test_set_used":False,
 "elapsed_seconds":time.perf_counter()-STARTED})
print("="*72);print("PHASE 05G CELL 4 — DECISION");print("="*72)
print("R2 ΔCIDEr vs R1           :",rows[2]["delta_cider_vs_r1"])
print("R3 ΔCIDEr vs R1           :",rows[3]["delta_cider_vs_r1"])
print("Selected arm              : R1")
print("New confirmation runs     : NOT AUTHORIZED")
print("Phase 5 conclusion        : RETAIN R1")
print("Progress                  : 4 / 5 code cells")


PHASE 05G CELL 4 — DECISION
R2 ΔCIDEr vs R1           : -0.003852951276902994
R3 ΔCIDEr vs R1           : -0.016004449611814797
Selected arm              : R1
New confirmation runs     : NOT AUTHORIZED
Phase 5 conclusion        : RETAIN R1
Progress                  : 4 / 5 code cells


## Cell 5 — Package Phase 5 conclusion and authorize Phase 6 entry


In [5]:
STARTED=time.perf_counter()
files=sorted(AD.glob("*.json"))+sorted(RD.glob("*"))
manifest=MD/"phase5g_manifest.csv";mrs=[]
for p in files:mrs.append({"relative_path":p.relative_to(OUT).as_posix(),"size_bytes":p.stat().st_size,"sha256":sha(p)})
with manifest.open("w",encoding="utf-8",newline="") as h:
    w=csv.DictWriter(h,fieldnames=["relative_path","size_bytes","sha256"]);w.writeheader();w.writerows(mrs)
sums=MD/"SHA256SUMS";sums.write_text("".join(f"{r['sha256']}  {r['relative_path']}\n" for r in mrs),encoding="utf-8")
auth=OUT/"PHASE6_MULTI_TEACHER_CONFIRMATION_ENTRY_AUTHORIZED.json"
write(auth,{"schema_version":"phase6_multi_teacher_entry_authorized_v1","created_at_utc":now(),
 "status":"authorized","phase5_selected_arm":"R1","selected_method":"Sequence KD alpha=0.5",
 "selected_source":"Phase 04F three-seed aggregate","new_phase5_confirmation_runs":False,
 "required_next":"Phase 6 multi-teacher confirmation entry and protocol lock",
 "test_set_status":"sealed","test_set_used":False})
marker=OUT/"PHASE5_SEQUENCE_KD_IMPROVEMENT_COMPLETE.json"
write(marker,{"schema_version":"phase5_sequence_kd_improvement_complete_v1","created_at_utc":now(),
 "status":"complete","feature_branch":"incompatible_stopped","weight_ablation":"complete",
 "selected_arm":"R1","selected_teacher_weight":0.5,"selected_human_weight":0.5,
 "r2_gate_passed":False,"r3_gate_passed":False,"new_confirmation_runs_authorized":False,
 "r1_three_seed_reproduction_status":R1A["reproduction_status"],
 "phase6_entry_authorized":True,
 "phase6_authorization_sha256":sha(auth),
 "manifest":{"csv":{"relative_path":manifest.relative_to(OUT).as_posix(),"sha256":sha(manifest),"rows":len(mrs)},
 "sha256sums":{"relative_path":sums.relative_to(OUT).as_posix(),"sha256":sha(sums)}},
 "training_started":False,"test_set_status":"sealed","test_set_used":False})
print("="*72);print("PHASE 05G FINAL STATUS");print("="*72)
print("Status                    : COMPLETE")
print("Selected arm              : R1 | alpha=0.5")
print("R2 / R3 gate              : FAILED / FAILED")
print("New confirmation runs     : NOT AUTHORIZED")
print("Existing R1 3-seed result : RETAINED")
print("Phase 6 entry             : AUTHORIZED")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 5 code cells")


PHASE 05G FINAL STATUS
Status                    : COMPLETE
Selected arm              : R1 | alpha=0.5
R2 / R3 gate              : FAILED / FAILED
New confirmation runs     : NOT AUTHORIZED
Existing R1 3-seed result : RETAINED
Phase 6 entry             : AUTHORIZED
Test Set status           : SEALED
Progress                  : 5 / 5 code cells
